In [2]:
# ============================================================
#            NEXT-GEN ADVANCED NLP TEXT ANALYZER
# ============================================================

# -----------------------------
# 1. INSTALL & UPGRADE LIBRARIES
# -----------------------------
!pip install -q nltk spacy vaderSentiment gradio matplotlib pandas textstat

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Ensure spaCy model is downloaded
!python -m spacy download en_core_web_sm -q

# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------
import re
import spacy
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import textstat
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# -----------------------------
# 3. INITIALIZE NLP TOOLS
# -----------------------------
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
sentiment_analyzer = SentimentIntensityAnalyzer()
nlp = spacy.load("en_core_web_sm")

# Set matplotlib to run headless
import matplotlib
matplotlib.use('Agg')

# -----------------------------
# 4. ADVANCED NLP ANALYSIS FUNCTION
# -----------------------------
def analyze_text_advanced(text):
    if not text or not text.strip():
        return (
            "Please enter some text to analyze.", # Overview
            None,                                  # Word Freq Plot
            "",                                    # Sentences
            "",                                    # Tokens Table
            "",                                    # Lemmas Table
            "",                                    # POS Tags
            "",                                    # NER
            ""                                     # Sentiment
        )

    # Basic Clean Up and Word stats
    sentences = sent_tokenize(text)
    words_all = word_tokenize(text)
    words_only = [w.lower() for w in words_all if w.isalnum()]

    char_count = len(text)
    word_count = len(words_only)
    sentence_count = len(sentences)
    avg_word_len = sum(len(w) for w in words_only) / max(1, word_count)

    # Readability Metric
    readability = textstat.flesch_reading_ease(text)
    readability_grade = textstat.text_standard(text)

    overview_html = f"""
    <div style='background-color: var(--background-fill-secondary); padding: 15px; border-radius: 8px; margin-bottom: 15px;'>
        <h3 style='margin-top: 0; color: var(--block-title-text-color);'>📊 Document Statistics</h3>
        <table style='width: 100%; border-collapse: collapse;'>
            <tr>
                <td><b>Words:</b> {word_count}</td>
                <td><b>Characters:</b> {char_count}</td>
                <td><b>Sentences:</b> {sentence_count}</td>
            </tr>
            <tr>
                <td><b>Avg. Word Length:</b> {avg_word_len:.2f} chars</td>
                <td><b>Readability Score:</b> {readability}</td>
                <td><b>Est. Grade Level:</b> {readability_grade}</td>
            </tr>
        </table>
    </div>
    """

    # Generate Word Frequency Chart
    freq_dist = Counter(words_only)
    common_words = freq_dist.most_common(10)

    fig, ax = plt.subplots(figsize=(6, 3))
    if common_words:
        words_labels, counts = zip(*common_words)
        ax.bar(words_labels, counts, color='#4F46E5')
        ax.set_title('Top 10 Most Frequent Words', fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        plt.tight_layout()
    else:
        ax.text(0.5, 0.5, 'No words found', ha='center', va='center')

    # Sentence Tokenization Output
    sentence_result = "\n".join(f"[{i+1}] {sentence.strip()}" for i, sentence in enumerate(sentences))

    # Stemming & Lemmatization Dataframe Representation
    table_data = []
    for word in words_only[:100]:  # Limit to first 100 words for preview speed
        stem = stemmer.stem(word)
        lemma = lemmatizer.lemmatize(word)
        table_data.append((word, stem, lemma))

    df_words = pd.DataFrame(table_data, columns=["Original Word", "Stem (Porter)", "Lemma (WordNet)"])
    stem_html = df_words[["Original Word", "Stem (Porter)"]].to_html(classes="pure-table", index=False)
    lemma_html = df_words[["Original Word", "Lemma (WordNet)"]].to_html(classes="pure-table", index=False)

    # POS Tagging & NER using spaCy
    doc = nlp(text)
    pos_tags = [f"{token.text} ({token.pos_})" for token in doc if not token.is_punct]
    pos_result = ", ".join(pos_tags[:150]) + ("..." if len(pos_tags) > 150 else "")

    # NER
    if doc.ents:
        ner_result = "\n".join(f"• {ent.text} ➔ {ent.label_} ({spacy.explain(ent.label_)})" for ent in doc.ents)
    else:
        ner_result = "No Named Entities detected."

    # Sentiment Analysis
    sentiment_scores = sentiment_analyzer.polarity_scores(text)
    compound = sentiment_scores["compound"]
    if compound >= 0.05:
        sentiment_label = "Positive 😊"
    elif compound <= -0.05:
        sentiment_label = "Negative 😞"
    else:
        sentiment_label = "Neutral 😐"

    sentiment_result = f"""### Analysis Result: {sentiment_label}

* **Compound Score:** {sentiment_scores['compound']:.4f}
* **Positive Score:** {sentiment_scores['pos']:.3f}
* **Neutral Score:** {sentiment_scores['neu']:.3f}
* **Negative Score:** {sentiment_scores['neg']:.3f}
"""

    return (
        overview_html,
        fig,
        sentence_result,
        stem_html,
        lemma_html,
        pos_result,
        ner_result,
        sentiment_result
    )

# -----------------------------
# 5. GRADIO USER INTERFACE DESIGN
# -----------------------------
with gr.Blocks() as demo_app:
    gr.Markdown(
        """
        # 🚀 Next-Gen NLP Workspace & Text Analyzer
        An advanced, multi-tiered natural language analysis toolkit engineered to analyze document structure, linguistic traits, and semantic context.
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="✍️ Source Text Input",
                placeholder="Type or paste your text here...",
                lines=10
            )
            analyze_btn = gr.Button("⚡ Run Full Pipeline", variant="primary")

        with gr.Column(scale=1):
            overview_output = gr.HTML(label="📊 Performance Metrics")
            plot_output = gr.Plot(label="📈 Term Frequency Map")

    gr.Markdown("--- ")

    # Organizing Outputs inside structured Tabs
    with gr.Tabs():
        with gr.TabItem("📝 Document Structure"):
            sentence_output = gr.Textbox(label="Sentence Boundary Detection (SBD)", lines=12)

        with gr.TabItem("🧬 Morphology & Lemmatization"):
            with gr.Row():
                stem_output = gr.HTML(label="Stemming Table")
                lemma_output = gr.HTML(label="Lemmatization Table")

        with gr.TabItem("🏷️ POS & Entities"):
            with gr.Row():
                with gr.Column():
                    pos_output = gr.Textbox(label="Grammatical Parts of Speech (POS)", lines=10)
                with gr.Column():
                    ner_output = gr.Textbox(label="Named Entity Recognition (NER)", lines=10)

        with gr.TabItem("❤️ Sentiment Workspace"):
            sentiment_output = gr.Markdown(label="Sentiment Intensity Metric")

    # Event Binding
    analyze_btn.click(
        fn=analyze_text_advanced,
        inputs=text_input,
        outputs=[
            overview_output,
            plot_output,
            sentence_output,
            stem_output,
            lemma_output,
            pos_output,
            ner_output,
            sentiment_output
        ]
    )

# -----------------------------
# 6. LAUNCH APPLICATION
# -----------------------------
demo_app.launch(share=True, theme=gr.themes.Default(primary_hue="indigo", secondary_hue="slate"))

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0383d3296689865216.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
